# Function Calling

### A. Introducción

In [7]:
def capital_Luna():

    info = "La Luna tiene su capital que es Lunar City !"

    return info

In [8]:
i = capital_Luna()
print(i)

La Luna tiene su capital que es Lunar City !


### Cargar el modelo

In [9]:
from langchain.chat_models import init_chat_model

from dotenv import load_dotenv
import os

# Generar una funcion que obtenga el modelo LLM

def get_llm(prefix_model, temperature=0.9):
    """
    Obtiene un modelo de lenguaje (LLM) basado en el modelo especificado.

    Args:
        model (str): El nombre del modelo a utilizar.
        temperature (float, optional): La temperatura para la generación de texto. 
                                        Valores más altos producen respuestas más creativas.
                                        Valores más bajos producen respuestas más deterministas.
                                        Por defecto es 0.9.

    Returns:
        llm: Un objeto de modelo de lenguaje inicializado.
    """
    llm = init_chat_model(
                            model=prefix_model, 
                            # temperature=temperature
                          )
    return llm

# Configuración del modelo de lenguaje
load_dotenv()

prefix_model = os.getenv("PREFIX_MODEL")

# Obtención del modelo
llm = get_llm(prefix_model)



### Usar el modelo

In [10]:
# 5. Usar el LLM

PREGUNTA = "Cual es la capital de la Luna?"

response = llm.invoke(PREGUNTA)

print(response.content)

[{'type': 'text', 'text': 'La Luna **no tiene capital**, ya que no está habitada por seres humanos ni cuenta con gobiernos, países o ciudades. \n\nSin embargo, desde el punto de vista científico y de la exploración espacial, hay lugares que han tenido un papel simbólico o de "centro de operaciones":\n\n* **Base Tranquilidad (Tranquility Base):** Es el lugar donde el ser humano pisó la Luna por primera vez (Apolo 11, en 1969). Podría considerarse la "capital histórica" de la presencia humana allí.\n* **El Polo Sur lunar:** Actualmente es considerado la "capital" científica y logística, ya que la mayoría de las misiones actuales y futuras (como el programa Artemis de la NASA) planean aterrizar allí debido a la presencia de hielo de agua.', 'extras': {'signature': 'EmAKXgFpFH0Te7xz5SrFej2//VYjozL18Ih+er+UPrWGqa5hIUFqlB/cQAio+Ar++wDJaFva+KDZpROBWXuVmoGowpBxNK99XZdtVoUUuEFhSP3unoNtLouI3qX6IJ9tsU0='}}]


### B. Pasos para crear una Function Calling

- Paso 1 : Definir una función
- Paso 2 : Definir la función como una herramienta
- Paso 3 : Vincularlo con el modelo

#### Paso 1

In [12]:
def capital_Luna():

    info = "La Luna tiene su capital que es Lunar City !"

    return info

#
print(capital_Luna)


<function capital_Luna at 0x1250f51c0>


#### Paso 2

In [16]:
from langchain.tools import tool

@tool
def capital_Luna():
    """
      Consulta informacion de la Luna, como puede ser la capital de la Luna
    """
    info = "La Luna tiene su capital que es Lunar City !"

    return info

#
print("Nombre:", capital_Luna.name)
print("Descripción:", capital_Luna.description)
print("Argumentos:", capital_Luna.args)

Nombre: capital_Luna
Descripción: Consulta informacion de la Luna, como puede ser la capital de la Luna
Argumentos: {}


#### Paso 3

In [20]:
from langchain_core.messages import HumanMessage
from langchain.agents import create_agent


agente_basico = create_agent(
    model = llm,
    tools = [
                capital_Luna
            ]
)

# Hacer la consulta

consulta = {"messages": [HumanMessage(content="Cual es la capital de la Luna?")]}

respuesta = agente_basico.invoke(consulta)

print(respuesta)



{'messages': [HumanMessage(content='Cual es la capital de la Luna?', additional_kwargs={}, response_metadata={}, id='124aac2a-9cfa-4f6d-98a1-c98f4ba99645'), AIMessage(content=[], additional_kwargs={'function_call': {'name': 'capital_Luna', 'arguments': '{}'}, '__gemini_function_call_thought_signatures__': {'call_3901558': 'EmAKXgFpFH0TaNq6EAidDFuNI0oOJdw/NrJ8PH33iI5pSpwywEJd5WR6G60KRLrySF5JHzjT1K0/cn+lSBqCePb7d+7t3mXCna7j2TtDolT2YAx/i2affnu+K2tSoEKXx4E='}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.5-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--01a14ab6-3dee-7081-8ff7-542248f27da6-0', tool_calls=[{'name': 'capital_Luna', 'args': {}, 'id': 'call_3901558', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 37, 'output_tokens': 10, 'total_tokens': 47, 'input_token_details': {'cache_read': 0}}), ToolMessage(content='La Luna tiene su capital que es Lunar City !', name='capital_Luna', id='991e4df0-af85-

In [22]:
    for m in respuesta["messages"]:
        print(f"[{type(m).__name__}] {m.text or m.tool_calls}")

[HumanMessage] Cual es la capital de la Luna?
[AIMessage] [{'name': 'capital_Luna', 'args': {}, 'id': 'call_3901558', 'type': 'tool_call'}]
[ToolMessage] La Luna tiene su capital que es Lunar City !
[AIMessage] La capital de la Luna es **Lunar City**.


### Ejercicio

Crear un herramienta nueva que me permita obtener información de la capital del planeta Marte

In [23]:
from langchain.tools import tool

@tool
def capital_Marte():
    """
      Consulta informacion del planeta Marte, como puede ser la capital de Marte
    """
    info = "El planeta marte tiene su capital que es Olimpus City !"

    return info


agente_basico = create_agent(
    model = llm,
    tools = [
                capital_Luna,
                capital_Marte
            ]
)

# Hacer la consulta

consulta = {"messages": [HumanMessage(content="Cual es la capital del planeta Marte?")]}

respuesta = agente_basico.invoke(consulta)

for m in respuesta["messages"]:
    print(f"[{type(m).__name__}] {m.text or m.tool_calls}")

[HumanMessage] Cual es la capital del planeta Marte?
[AIMessage] [{'name': 'capital_Marte', 'args': {}, 'id': 'call_2534064', 'type': 'tool_call'}]
[ToolMessage] El planeta marte tiene su capital que es Olimpus City !
[AIMessage] La capital del planeta Marte es **Olimpus City**.


### C. Pasos para crear una Function Calling con parámetros


In [ ]:
@tool
def info_Marte(consulta: str ) -> str:
    """
      Consulta la informacion del planeta Marte, usalo cuando te pregunten sobre
      - capital
      - población 
      - pbi
    """
    info = {
        "capital" :"La capital es Olimpus City !",
        "poblacion" : "1 habitante",
        "pbi" : " 1 sol" 
    }

    consulta = consulta.lower()
    print(f"consulta {consulta}")
    print(f"dato 1 {info.get(consulta)}")

    return info.get(consulta)



print("Nombre:", info_Marte.name)
print("Descripción:", info_Marte.description)
print("Argumentos:", info_Marte.args)

Nombre: info_Marte
Descripción: Consulta la informacion del planeta Marte, usalo cuando te pregunten sobre
- capital
- población 
- pbi
Argumentos: {'consulta': {'title': 'Consulta', 'type': 'string'}}


In [54]:
agente_basico = create_agent(
    model = llm,
    tools = [
                capital_Luna,
                info_Marte
            ]
)

In [55]:
# Hacer la consulta

consulta = {"messages": [HumanMessage(content="Cual es la población del planeta Marte?")]}

respuesta = agente_basico.invoke(consulta)

for m in respuesta["messages"]:
    print(f"[{type(m).__name__}] {m.text or m.tool_calls}")

consulta poblacion
dato 1 1 habitante
[HumanMessage] Cual es la población del planeta Marte?
[AIMessage] [{'name': 'info_Marte', 'args': {'consulta': 'poblacion'}, 'id': 'call_2810027', 'type': 'tool_call'}]
[ToolMessage] 1 habitante
[AIMessage] La población del planeta Marte es de 1 habitante.
